In [1]:
import geopandas as gpd

dist = gpd.read_file("2011_Dist.shp")
print("Districts:", len(dist), "| CRS:", dist.crs)
print("Columns:", list(dist.columns))
print(dist.drop(columns="geometry").head())

osm = gpd.read_file("osm_bbox_chargers.geojson")
print("\nOSM features:", len(osm))
print("Geometry types:", osm.geometry.geom_type.value_counts().to_dict())
print("OSM columns:", list(osm.columns)[:25])

Districts: 641 | CRS: EPSG:4326
Columns: ['DISTRICT', 'ST_NM', 'ST_CEN_CD', 'DT_CEN_CD', 'censuscode', 'geometry']
     DISTRICT           ST_NM  ST_CEN_CD  DT_CEN_CD  censuscode
0    Adilabad  Andhra Pradesh         28          1         532
1        Agra   Uttar Pradesh          9         15         146
2   Ahmadabad         Gujarat         24          7         474
3  Ahmadnagar     Maharashtra         27         26         522
4      Aizawl         Mizoram         15          3         283

OSM features: 230
Geometry types: {'Point': 230}
OSM columns: ['id', '@id', 'access', 'addr:city', 'addr:housenumber', 'addr:place', 'addr:street', 'amenity', 'authentication:nfc', 'bicycle', 'branch', 'brand', 'brand:en', 'brand:website', 'brand:wikidata', 'building', 'bus', 'capacity', 'capacity:motorcar', 'charge', 'charging_station:battery_swapping', 'charging_station:output', 'check_date', 'covered', 'description']


In [2]:
# 1. UP districts only
up = dist[dist["ST_NM"] == "Uttar Pradesh"].copy()
print("UP districts:", len(up))

# 2. Spatial join: keep only chargers inside UP
osm_up = gpd.sjoin(osm, up[["DISTRICT", "censuscode", "geometry"]],
                   how="inner", predicate="within")
print("Raw OSM records in bounding box:", len(osm))
print("Inside UP districts:", len(osm_up))
print("Dropped (outside UP):", len(osm) - len(osm_up))

# 3. Chargers per district
counts = osm_up.groupby("DISTRICT").size().sort_values(ascending=False)
print("\nDistricts with at least one charger:", len(counts), "of", len(up))
print(counts.head(15))

# 4. Which columns can help with dedupe?
for c in ["name", "operator", "brand", "network", "capacity", "access", "fee"]:
    if c in osm_up.columns:
        print(f"{c}: {osm_up[c].notna().sum()} filled of {len(osm_up)}")

UP districts: 71
Raw OSM records in bounding box: 230
Inside UP districts: 24
Dropped (outside UP): 206

Districts with at least one charger: 11 of 71
DISTRICT
Gautam Buddha Nagar    10
Agra                    3
Kanpur Nagar            2
Mathura                 2
Aligarh                 1
Allahabad               1
Fatehpur                1
Bijnor                  1
Lucknow                 1
Muzaffarnagar           1
Varanasi                1
dtype: int64
name: 7 filled of 24
operator: 6 filled of 24
brand: 7 filled of 24
capacity: 6 filled of 24
access: 3 filled of 24
fee: 2 filled of 24


In [3]:
from sklearn.cluster import DBSCAN
import numpy as np

# project to metres (UTM zone 44N covers most of UP)
pts = osm_up.to_crs(32644).copy()
xy = np.column_stack([pts.geometry.x, pts.geometry.y])

# group points within 100 m of each other
pts["cluster"] = DBSCAN(eps=100, min_samples=1).fit(xy).labels_

print("Raw records inside UP:", len(pts))
print("Unique chargers after 100 m merge:", pts["cluster"].nunique())

# show the merged groups for manual review
dups = pts[pts.duplicated("cluster", keep=False)].sort_values("cluster")
cols = [c for c in ["cluster", "DISTRICT", "name", "operator", "brand"] if c in dups.columns]
print(dups[cols] if len(dups) else "No points within 100 m of each other")

# keep one record per cluster
unique = pts.drop_duplicates("cluster").to_crs(4326)
print("Final unique chargers:", len(unique))

Raw records inside UP: 24
Unique chargers after 100 m merge: 22
     cluster             DISTRICT  name operator brand
177       17              Mathura  None     None  None
178       17              Mathura  None     None  None
179       18  Gautam Buddha Nagar  None     Tata  None
180       18  Gautam Buddha Nagar  None     None  None
Final unique chargers: 22


In [4]:
import pandas as pd, glob

print("CSV files here:", glob.glob("*.csv"))

for f in glob.glob("*.csv"):
    print("\n=====", f)
    try:
        df = pd.read_csv(f)
        print(df.shape)
        print(df.head(8).to_string())
    except Exception as e:
        print("could not read:", e)

CSV files here: ['RS_Session_267_AU_3929_E.ii_.csv', 'RS_Session_258_AU_1241_3.csv', 'RS_Session_267_AU_3266_2_0.csv', 'vahan_UP_ev_RTOwise_2018-2026_incl-archive_pulled_18Oct2026.csv']

===== RS_Session_267_AU_3929_E.ii_.csv
(4, 3)
                       Year  No. of PCS  No. of Newly Reported PCS
0  As on 31st December 2022        5151                       5151
1  As on 31st December 2023       11903                       6752
2  As on 31st December 2024       25202                      13299
3     As on 01st April 2025       26367                       1165

===== RS_Session_258_AU_1241_3.csv
(16, 3)
  Category   City/Highway  Charging Stations
0     City     Chandigarh                 48
1     City          Delhi                 94
2     City      Rajasthan                 49
3     City      Karnataka                 65
4     City      Jharkhand                 30
5     City            Goa                 30
6     City      Telangana                 57
7     City  Uttar Pradesh   

In [5]:
import pandas as pd, re, difflib

# 1. Official UP charger count
st = pd.read_csv("RS_Session_267_AU_3266_2_0.csv")
print(st[st.iloc[:, 1].str.contains("Uttar", case=False, na=False)].to_string())

# 2. VAHAN: first column holds the category names
fn = [f for f in __import__("glob").glob("vahan_UP_ev_RTOwise*.csv")][0]
v = pd.read_csv(fn)
v = v.rename(columns={v.columns[0]: "category"}).set_index("category")
v.columns = [c.strip() for c in v.columns]
print("\nCategories:", v.index.tolist())

rto_cols = [c for c in v.columns if c != "Total"]
print("RTO columns:", len(rto_cols))

# 3. Draft RTO -> district matching by name similarity
up_names = sorted(up["DISTRICT"].tolist())
print("\nUP districts in shapefile:\n", ", ".join(up_names))

def clean(c):
    n = re.sub(r"\s*-\s*UP\d+\s*$", "", c)
    n = re.sub(r"\(.*?\)", "", n)
    n = re.sub(r"\b(RTO|ARTO|OFFICE|ARTO OFFICE)\b", "", n, flags=re.I)
    return n.strip().title()

print("\nRTO -> best district guess (score):")
for c in rto_cols:
    n = clean(c)
    m = difflib.get_close_matches(n, up_names, n=1, cutoff=0.0)[0]
    s = difflib.SequenceMatcher(None, n.lower(), m.lower()).ratio()
    print(f"{c[:45]:45} -> {m} ({s:.2f})")

   Sl. No.       State/UT  No. of Electric Vehicle (EV) Charging Stations Installed
33      34  Uttar Pradesh                                                      2901
34      35    Uttarakhand                                                       252

Categories: ['Ambulance/Hearses', 'Bus', 'Construction Equipment Vehicle', 'Four Wheeler', 'Goods Vehicle', 'Public Service Vehicle', 'Special Category Vehicles', 'Three Wheeler', 'Tractor', 'Trailer', 'Two Wheeler', 'Total']
RTO columns: 76

UP districts in shapefile:
 Agra, Aligarh, Allahabad, Ambedkar Nagar, Auraiya, Azamgarh, Baghpat, Bahraich, Ballia, Balrampur, Banda, Bara Banki, Bareilly, Basti, Bijnor, Budaun, Bulandshahr, Chandauli, Chitrakoot, Deoria, Etah, Etawah, Faizabad, Farrukhabad, Fatehpur, Firozabad, Gautam Buddha Nagar, Ghaziabad, Ghazipur, Gonda, Gorakhpur, Hamirpur, Hardoi, Jalaun, Jaunpur, Jhansi, Jyotiba Phule Nagar, Kannauj, Kanpur Dehat, Kanpur Nagar, Kansiram Nagar, Kaushambi, Kheri, Kushinagar, Lalitpur, Luckno

In [6]:
import re, difflib, pandas as pd

# manual overrides by RTO code
over = {
 "UP42":"Faizabad", "UP16":"Gautam Buddha Nagar", "UP70":"Allahabad",
 "UP32":"Lucknow", "UP321":"Lucknow", "UP45":"Ambedkar Nagar",
 "UP36":"Sultanpur", "UP66":"Sant Ravi Das Nagar(bhadohi)", "UP37":"Ghaziabad",
 "UP86":"Mahamaya Nagar", "UP23":"Jyotiba Phule Nagar", "UP87":"Kansiram Nagar",
 "UP31":"Kheri", "UP92":"Jalaun", "UP57":"Kushinagar", "UP33":"Rae Bareli",
 "UP19":"Muzaffarnagar", "UP38":"Moradabad", "UP46":"Shrawasti", "UP90":"Banda",
}
post2011 = {"UP36","UP37","UP19","UP38"}   # approximate parent mapping

rows = []
for c in rto_cols:
    m = re.search(r"(UP\d+)\s*$", c)
    code = m.group(1) if m else None
    if code in over:
        d, how = over[code], "manual"
    else:
        n = clean(c)
        d = difflib.get_close_matches(n, up_names, n=1, cutoff=0.0)[0]
        s = difflib.SequenceMatcher(None, n.lower(), d.lower()).ratio()
        how = "auto" if s >= 0.8 else f"CHECK({s:.2f})"
    rows.append({"rto": c, "code": code, "district": d, "method": how,
                 "approx_post2011": code in post2011})
mp = pd.DataFrame(rows)
print("Needs checking:\n", mp[mp.method.str.startswith("CHECK")])
mp.to_csv("rto_district_mapping.csv", index=False)

num = lambda s: pd.to_numeric(s.astype(str).str.replace(",", ""), errors="coerce").fillna(0)
ev4 = num(v.loc["Four Wheeler", mp.rto])
ev3 = num(v.loc["Three Wheeler", mp.rto])
mp["ev4"], mp["ev3"] = ev4.values, ev3.values

d = mp.groupby("district")[["ev4","ev3"]].sum()
ch = unique.groupby("DISTRICT").size().rename("chargers")
d = d.join(ch).fillna({"chargers": 0})
d["chargers"] = d["chargers"].astype(int)
d["ev4_per_charger"] = d["ev4"] / d["chargers"].clip(lower=1)   # zero treated as 1 (lower bound)

print("\nUP 4W EVs mapped:", int(d.ev4.sum()), "| districts without RTO:",
      sorted(set(up_names) - set(d.index)))
print("OSM share of official:", f"{len(unique)}/2901 = {len(unique)/2901:.1%}")

top10 = d.sort_values("ev4_per_charger", ascending=False).head(10)
print("\nTop 10 most underserved (4W EVs per mapped charger):")
print(top10.round(0).astype({"ev4":int,"ev3":int}).to_string())
top10.to_csv("top10_underserved.csv")

Needs checking:
 Empty DataFrame
Columns: [rto, code, district, method, approx_post2011]
Index: []

UP 4W EVs mapped: 43599 | districts without RTO: []
OSM share of official: 22/2901 = 0.8%

Top 10 most underserved (4W EVs per mapped charger):
                      ev4    ev3  chargers  ev4_per_charger
district                                                   
Lucknow              8704  99554         1           8704.0
Ghaziabad            5154  53301         0           5154.0
Meerut               2098  26440         0           2098.0
Kanpur Nagar         2856  76213         2           1428.0
Allahabad            1289  49431         1           1289.0
Varanasi             1265  32081         1           1265.0
Gautam Buddha Nagar  9559  53599         9           1062.0
Gorakhpur            1007  26904         0           1007.0
Bareilly              882  36271         0            882.0
Mathura               772  32831         1            772.0
